# 04 - Enriquecimiento con distancia a estaciones de subte

## Objetivo
Para cada aviso inmobiliario con coordenadas (lat/lon), calcular:
- La **estación más cercana** y su distancia
- La **distancia mínima a cada línea** de subte

## Entradas
- `backup_geocoding.csv` — dataset con columnas `lat` y `lon`
- `estaciones_subte.csv` — estaciones con geometría WKT (POINT)

## Salidas
- `outputs/interm/04_df_distancia_subte.csv`


## Librerías y parámetros

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np
from shapely import wkt

out_interm = Path('outputs/interm')
out_interm.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

## 1. Función de distancia

Usamos la fórmula de **Haversine**, que calcula la distancia real entre dos puntos sobre la esfera terrestre a partir de sus coordenadas lat/lon. El resultado está en **metros**.

No requiere librerías externas como GeoPandas o pyproj.

In [20]:
def haversine_metros(lat1, lon1, lat2, lon2):
    """
    Calcula la distancia en metros entre dos puntos geográficos.
    Acepta arrays de numpy (vectorizado) o valores individuales.
    """
    R = 6_371_000  # radio de la Tierra en metros

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Test rápido: distancia entre dos puntos conocidos de CABA
d = haversine_metros(-34.603722, -58.381592, -34.615748, -58.398930)
print(f'Test Haversine: {d:.0f} metros (debería ser ~2 km aprox)')

Test Haversine: 2075 metros (debería ser ~2 km aprox)


## 2. Cargar estaciones de subte

In [21]:
df_subte = pd.read_csv('estaciones_de_subte.csv')


# Parsear la columna geometry (formato WKT: "POINT (lon lat)")
df_subte['geom'] = df_subte['geometry'].apply(wkt.loads)
df_subte['est_lon'] = df_subte['geom'].apply(lambda p: p.x)
df_subte['est_lat'] = df_subte['geom'].apply(lambda p: p.y)

lineas = sorted(df_subte['linea'].unique())
print(f'Estaciones cargadas: {len(df_subte)}')
print(f'Líneas: {lineas}')
display(df_subte[['id', 'estacion', 'linea', 'est_lat', 'est_lon']].head(5))


Estaciones cargadas: 90
Líneas: ['A', 'B', 'C', 'D', 'E', 'H']


,id,estacion,linea,est_lat,est_lon
0,1,Caseros,H,-34.635749,-58.398930
1,2,Inclan - Mezquita Al Ahmad,H,-34.629374,-58.400972
2,3,Humberto 1°,H,-34.623091,-58.402325
3,4,Venezuela,H,-34.615241,-58.404734
4,5,Once - 30 De Diciembre,H,-34.608934,-58.406039


## 3. Cargar dataset de avisos con coordenadas

In [22]:
df = pd.read_csv('backup_geocoding.csv')

# Asegurar que lat/lon sean numéricos
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce')

n_total = len(df)
n_con_coords = df[['lat', 'lon']].dropna().shape[0]

print(f'Filas totales:         {n_total}')
print(f'Con coordenadas:       {n_con_coords}')
print(f'Sin coordenadas (NaN): {n_total - n_con_coords}')

Filas totales:         104084
Con coordenadas:       83326
Sin coordenadas (NaN): 20758


## 4. Calcular distancias

Para cada aviso calculamos la distancia a **todas** las estaciones usando la función vectorizada de Haversine.
Esto es mucho más rápido que iterar fila por fila con `.apply()`.

Luego extraemos:
- La estación más cercana (global)
- La distancia mínima a cada línea

In [23]:
# Arrays de coordenadas de las estaciones (shape: n_estaciones)
est_lats = df_subte['est_lat'].values
est_lons = df_subte['est_lon'].values
est_nombres = df_subte['estacion'].values
est_lineas = df_subte['linea'].values

# Separar avisos con y sin coordenadas
mask_validos = df['lat'].notna() & df['lon'].notna()
df_validos = df[mask_validos].copy()

print(f'Procesando {len(df_validos)} avisos con coordenadas...')

# Matriz de distancias: shape (n_avisos, n_estaciones)
# Cada fila i tiene la distancia del aviso i a cada estación
av_lats = df_validos['lat'].values[:, np.newaxis]  # columna
av_lons = df_validos['lon'].values[:, np.newaxis]  # columna

dist_matrix = haversine_metros(av_lats, av_lons, est_lats, est_lons)
# dist_matrix[i, j] = distancia del aviso i a la estación j

print(f'Matriz de distancias calculada: {dist_matrix.shape} (avisos x estaciones)')

Procesando 83326 avisos con coordenadas...
Matriz de distancias calculada: (83326, 90) (avisos x estaciones)


In [24]:
# --- Estación más cercana (global) ---
idx_min = np.argmin(dist_matrix, axis=1)  # índice de la estación más cercana por aviso

df_validos['estacion_cercana']          = est_nombres[idx_min]
df_validos['linea_cercana']             = est_lineas[idx_min]
df_validos['distancia_estacion_cercana'] = np.round(dist_matrix[np.arange(len(df_validos)), idx_min], 1)

# --- Distancia mínima a cada línea ---
for linea in lineas:
    mask_linea = est_lineas == linea          # columnas de esa línea
    dist_linea = dist_matrix[:, mask_linea]  # sub-matriz solo con esas estaciones
    df_validos[f'distancia_linea_{linea}'] = np.round(dist_linea.min(axis=1), 1)

print('Columnas nuevas generadas:')
cols_nuevas = ['estacion_cercana', 'linea_cercana', 'distancia_estacion_cercana'] + \
              [f'distancia_linea_{l}' for l in lineas]
print(cols_nuevas)

Columnas nuevas generadas:
['estacion_cercana', 'linea_cercana', 'distancia_estacion_cercana', 'distancia_linea_A', 'distancia_linea_B', 'distancia_linea_C', 'distancia_linea_D', 'distancia_linea_E', 'distancia_linea_H']


## 5. Verificar resultados

In [25]:
print('Muestra de resultados:')
cols_muestra = ['lat', 'lon'] + cols_nuevas
display(df_validos[cols_muestra].head(10))

print('\nEstadísticas de distancia_estacion_cercana (metros):')
print(df_validos['distancia_estacion_cercana'].describe().round(1))

print('\nEstaciones más frecuentes como cercanas:')
print(df_validos['estacion_cercana'].value_counts().head(10))

Muestra de resultados:


,lat,lon,estacion_cercana,linea_cercana,distancia_estacion_cercana,distancia_linea_A,distancia_linea_B,distancia_linea_C,distancia_linea_D,distancia_linea_E,distancia_linea_H
0,-34.553253,-58.449215,Juramento,D,1207.4,6973.4,3766.6,8020.7,1207.4,8000.4,6093.7
1,-34.704028,-58.495748,Plaza De Los Virreyes - Eva Perón,E,7436.7,8495.5,12755.1,13471.5,14645.7,7436.7,10335.1
4,-34.565938,-58.448770,Jose Hernandez,D,308.9,5662.0,2432.4,7257.5,308.9,7037.1,5291.8
6,-34.590645,-58.415321,Bulnes,D,455.4,2239.0,1484.8,3467.3,455.4,3628.6,1260.2
7,-34.582385,-58.425019,Plaza Italia,D,366.2,3280.5,2272.4,4544.3,366.2,4642.0,2469.0
8,-34.590969,-58.423125,R.Scalabrini Ortiz,D,921.0,2316.0,1370.4,4171.6,921.0,3887.2,1941.4
9,-34.578115,-58.449303,Olleros,D,995.8,4504.9,1138.0,6809.3,995.8,5726.9,4665.5
10,-34.553995,-58.459472,Congreso De Tucuman,D,323.2,7290.7,3300.9,8759.6,323.2,8559.3,6807.2
11,-34.552571,-58.442991,Juramento,D,1643.4,6862.0,4005.0,7610.4,1643.4,7571.4,5711.0
12,-34.595482,-58.421248,Almagro - Medrano,B,854.7,1789.3,854.7,3957.4,1216.1,3363.8,1637.7



Estadísticas de distancia_estacion_cercana (metros):
count      83326.0
mean       27280.6
std        97530.9
min            1.5
25%          334.7
50%          668.7
75%         1790.3
max      4867277.9
Name: distancia_estacion_cercana, dtype: float64

Estaciones más frecuentes como cercanas:
estacion_cercana
Juan Manuel De Rosas - Villa Urquiza    7224
Congreso De Tucuman                     6219
San Pedrito                             4650
Plaza De Los Virreyes - Eva Perón       4265
Ministro Carranza - Miguel Abuelo       3787
R.Scalabrini Ortiz                      2875
Hospitales - Ringo Bonavena             2680
Juramento                               2503
Olleros                                 2493
Palermo                                 2383
Name: count, dtype: int64


## 6. Reunir con filas sin coordenadas y exportar

In [26]:
# Reunir: filas con coords (enriquecidas) + filas sin coords (NaN en columnas nuevas)
df_sin_coords = df[~mask_validos].copy()
for col in cols_nuevas:
    df_sin_coords[col] = np.nan

df_final = pd.concat([df_validos, df_sin_coords], ignore_index=True)

# Exportar
out_path = out_interm / '02_4_df_distancia_subte.csv'
df_final.to_csv(out_path, index=False)

print(f'Filas exportadas: {len(df_final)}')
print(f'Columnas totales: {df_final.shape[1]}')
print(f'Guardado en: {out_path}')

Filas exportadas: 104084
Columnas totales: 45
Guardado en: outputs/interm/02_4_df_distancia_subte.csv
